# Combinatorial Interface Sets for Semiconductor Electron-Phonon Transport

Generate sets of interfaces from combinations of substrate and film materials over a list of Miller indices. Each
set pairs a list of substrates with a list of films, for example semiconductors with gate dielectrics or
semiconductors with metal contacts. Every combination is matched with the Zur and McGill superlattice
[algorithm](https://doi.org/10.1063/1.3330840) and kept only if a supercell exists within the strain and size limits.

The default sets cover interfaces where electron-phonon physics limits device behavior:
1. Semiconductor heterostructures: strained channels and 2D electron gases (polar-optical and interface phonons).
1. Semiconductor / gate dielectric: remote (surface-optical) phonon scattering from the dielectric.
1. Semiconductor / metal: contacts and heat spreaders (thermal boundary conductance, electron-phonon non-equilibrium).
1. 2D materials on substrates: substrate phonon scattering and encapsulation of 2D channels.

<h2 style="color:green">Usage</h2>

1. Set the material lists, the sets, and the limits in cell 1.1.
1. Click "Run" > "Run All" to run all cells.
1. Review the screening tables in section 2 and the generated interfaces in sections 3 and 4.
1. Each set is saved to `uploads/<set name>/`. Use [Create Material Set](utils_create_material_set.ipynb) with `SUBFOLDER_NAME` set to the set name to upload it to the platform as a material set.

## Summary
1. Prepare the Environment: define material lists and sets, load materials
1. Screen all substrate/film/Miller index combinations of each set with the ZSL analyzer
1. Create the interfaces that pass the strain and size limits
1. Preview and save each set

## Notes
1. Miller indices refer to the conventional cell (`(0, 0, 1)` is the basal plane of hexagonal crystals).
1. 2D materials are used as single layers on the `(0, 0, 1)` plane, with a van der Waals gap to the neighboring material.
1. For each combination, the match with the fewest atoms is selected among those within the strain and atom limits; ties are resolved by lower strain.
1. The first slab termination is used for both film and substrate.
1. Gate dielectrics are represented by crystalline phases.
1. Materials not available in Standata are built from crystal prototypes with experimental lattice constants.

## 1. Prepare the Environment
### 1.1. Set up material lists, sets, and limits

Materials are Standata names or keys of `PROTOTYPE_MATERIALS`. Each set takes `substrates` and `films`, and can
override the defaults with `miller_indices`, `max_strain_percentage`, `max_number_of_atoms`, and `max_area`.

In [ ]:
SEMICONDUCTORS = [
    "Silicon, FCC",
    "Germanium",
    "Gallium Arsenide",
    "AlAs",
    "Gallium Nitride",
    "Aluminum Nitride",
    "Diamond",
]
DIELECTRICS = [
    "Sapphire",
    "Cristobalite",
    "HfO2, Hafnium IV Oxide, MCL",
    "Zirconium Dioxide",
]
METALS = [
    "Aluminum, FCC",
    "Copper",
    "Gold",
    "Platinum",
    "Nickel",
    "Titanium Nitride",
]
TWO_DIMENSIONAL_MATERIALS = [
    "Graphene, HEX",
    "Hexagonal Boron Nitride",
    "Molybdenum Disulfide, HEX (P-6m2)",
    "Tungsten Diselenide, HEX (P-6m2)",
]

# Lattice constants in Angstrom
PROTOTYPE_MATERIALS = {
    "AlAs": {"crystalstructure": "zincblende", "a": 5.661},
}

SETS = {
    "semiconductor_heterostructures": {
        "substrates": SEMICONDUCTORS,
        "films": SEMICONDUCTORS,
    },
    "semiconductor_dielectric": {
        "substrates": SEMICONDUCTORS,
        "films": DIELECTRICS,
        "max_number_of_atoms": 300,
        "max_area": 250,
    },
    "semiconductor_metal": {
        "substrates": SEMICONDUCTORS,
        "films": METALS,
    },
    "two_dimensional_on_substrates": {
        "substrates": ["Hexagonal Boron Nitride", "Sapphire", "Silicon, FCC", "Gallium Nitride", "Aluminum Nitride"],
        "films": TWO_DIMENSIONAL_MATERIALS,
    },
}

MILLER_INDICES = [(0, 0, 1), (1, 1, 1)]
# Optional per-material Miller indices, e.g. {"Nickel": [(1, 1, 1)]}
MILLER_INDICES_BY_MATERIAL = {}

MAX_STRAIN_PERCENTAGE = 3.0
MAX_NUMBER_OF_ATOMS = 200

SUBSTRATE_THICKNESS = 2  # in atomic layers
FILM_THICKNESS = 2  # in atomic layers
VAN_DER_WAALS_GAP = 3.35  # in Angstrom, used when either material is 2D
INTERFACE_VACUUM = 10.0  # in Angstrom

# Supercell search parameters
MAX_AREA = 150  # in Angstrom^2
MAX_AREA_TOLERANCE = 0.09
MAX_LENGTH_TOLERANCE = 0.05
MAX_ANGLE_TOLERANCE = 0.02

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Load materials

In [ ]:
from ase.build import bulk
from mat3ra.made.material import Material
from mat3ra.made.tools.convert import from_ase
from mat3ra.standata.materials import Materials


def get_material(name):
    if name in PROTOTYPE_MATERIALS:
        material = Material.create(from_ase(bulk(name, **PROTOTYPE_MATERIALS[name])))
        material.name = name
        return material
    return Material.create(Materials.get_by_name_first_match(name))


material_names = list(
    dict.fromkeys(
        name for set_definition in SETS.values() for key in ("substrates", "films") for name in set_definition[key]
    )
)
materials = {name: get_material(name) for name in material_names}

### 1.4. Preview materials

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize

visualize([{"material": material, "title": name} for name, material in materials.items()], repetitions=[2, 2, 1])

## 2. Screen combinations
### 2.1. Enumerate substrate/film/Miller index combinations for each set

In [ ]:
import itertools


def get_set_parameters(set_definition):
    return {
        "miller_indices": set_definition.get("miller_indices", MILLER_INDICES),
        "max_strain_percentage": set_definition.get("max_strain_percentage", MAX_STRAIN_PERCENTAGE),
        "max_number_of_atoms": set_definition.get("max_number_of_atoms", MAX_NUMBER_OF_ATOMS),
        "matching_parameters": {
            "max_area": set_definition.get("max_area", MAX_AREA),
            "max_area_ratio_tol": MAX_AREA_TOLERANCE,
            "max_length_tol": MAX_LENGTH_TOLERANCE,
            "max_angle_tol": MAX_ANGLE_TOLERANCE,
        },
    }


def get_miller_indices(name, set_parameters):
    if name in TWO_DIMENSIONAL_MATERIALS:
        return [(0, 0, 1)]
    return MILLER_INDICES_BY_MATERIAL.get(name, set_parameters["miller_indices"])


set_parameters_by_name = {set_name: get_set_parameters(set_definition) for set_name, set_definition in SETS.items()}

combinations = []
for set_name, set_definition in SETS.items():
    set_parameters = set_parameters_by_name[set_name]
    for substrate_name, film_name in itertools.product(set_definition["substrates"], set_definition["films"]):
        if substrate_name == film_name:
            continue
        for substrate_miller_indices in get_miller_indices(substrate_name, set_parameters):
            for film_miller_indices in get_miller_indices(film_name, set_parameters):
                combinations.append((set_name, substrate_name, substrate_miller_indices, film_name, film_miller_indices))

for set_name in SETS:
    print(f"{set_name}: {sum(combination[0] == set_name for combination in combinations)} combinations")

### 2.2. Find the best match for each combination

In [ ]:
import numpy as np
from mat3ra.made.tools.analyze.interface import ZSLInterfaceAnalyzer
from mat3ra.made.tools.build.pristine_structures.two_dimensional.slab import SlabBuilder, SlabConfiguration

slabs = {}


def get_slab(name, miller_indices, number_of_layers):
    if name in TWO_DIMENSIONAL_MATERIALS:
        number_of_layers = 1
    key = (name, miller_indices, number_of_layers)
    if key not in slabs:
        configuration = SlabConfiguration.from_parameters(
            material_or_dict=materials[name],
            miller_indices=miller_indices,
            number_of_layers=number_of_layers,
            vacuum=0.0,
            use_conventional_cell=True,
        )
        slabs[key] = (configuration, SlabBuilder().get_material(configuration))
    return slabs[key]


def count_atoms(match, substrate_slab, film_slab):
    substrate_cells = abs(np.linalg.det(match.substrate_supercell_matrix.model_dump()))
    film_cells = abs(np.linalg.det(match.film_supercell_matrix.model_dump()))
    return round(
        substrate_cells * len(substrate_slab.basis.elements.values)
        + film_cells * len(film_slab.basis.elements.values)
    )


def find_best_match(set_parameters, substrate_name, substrate_miller_indices, film_name, film_miller_indices):
    substrate_configuration, substrate_slab = get_slab(substrate_name, substrate_miller_indices, SUBSTRATE_THICKNESS)
    film_configuration, film_slab = get_slab(film_name, film_miller_indices, FILM_THICKNESS)
    zsl_analyzer = ZSLInterfaceAnalyzer(
        substrate_slab_configuration=substrate_configuration,
        film_slab_configuration=film_configuration,
        reduce_result_cell=False,
        **set_parameters["matching_parameters"],
    )
    candidates = []
    for match_index, match in enumerate(zsl_analyzer.zsl_match_holders):
        number_of_atoms = count_atoms(match, substrate_slab, film_slab)
        if (
            match.total_strain_percentage <= set_parameters["max_strain_percentage"]
            and number_of_atoms <= set_parameters["max_number_of_atoms"]
        ):
            candidates.append((number_of_atoms, match.total_strain_percentage, match_index, match.match_area))
    return min(candidates) if candidates else None


screening_rows = []
for set_name, substrate_name, substrate_miller_indices, film_name, film_miller_indices in combinations:
    best_match = find_best_match(
        set_parameters_by_name[set_name], substrate_name, substrate_miller_indices, film_name, film_miller_indices
    )
    row = {
        "set": set_name,
        "substrate": substrate_name,
        "substrate miller": substrate_miller_indices,
        "film": film_name,
        "film miller": film_miller_indices,
        "accepted": best_match is not None,
    }
    if best_match:
        number_of_atoms, strain_percentage, match_index, match_area = best_match
        row.update(
            {
                "strain, %": round(strain_percentage, 2),
                "area, Angstrom^2": round(match_area, 1),
                "number of atoms": number_of_atoms,
                "match index": match_index,
            }
        )
    screening_rows.append(row)

### 2.3. Review the screening results

In [ ]:
import pandas as pd

screening_table = pd.DataFrame(screening_rows)
screening_table.groupby("set", sort=False)["accepted"].agg(combinations="size", accepted="sum")

In [ ]:
accepted_table = screening_table[screening_table["accepted"]].drop(columns=["accepted"])
accepted_table = accepted_table.astype({"number of atoms": int, "match index": int})
accepted_table["index in set"] = accepted_table.groupby("set").cumcount()
accepted_table.set_index(["set", "index in set"])

## 3. Create the interfaces
The interfacial distance is estimated from the bulk radial distribution functions, or set to `VAN_DER_WAALS_GAP`
when either material is 2D.

In [ ]:
from mat3ra.made.tools.analyze.interface import calculate_interfacial_distance_from_rdf
from mat3ra.made.tools.helpers import create_interface_zsl_between_slabs

gaps = {}


def get_gap(substrate_name, film_name):
    if substrate_name in TWO_DIMENSIONAL_MATERIALS or film_name in TWO_DIMENSIONAL_MATERIALS:
        return VAN_DER_WAALS_GAP
    key = (substrate_name, film_name)
    if key not in gaps:
        gaps[key] = calculate_interfacial_distance_from_rdf(
            substrate_material=materials[substrate_name], film_material=materials[film_name]
        )
    return gaps[key]


interfaces_by_set = {set_name: [] for set_name in SETS}
for _, row in accepted_table.iterrows():
    _, substrate_slab = get_slab(row["substrate"], row["substrate miller"], SUBSTRATE_THICKNESS)
    _, film_slab = get_slab(row["film"], row["film miller"], FILM_THICKNESS)
    interface = create_interface_zsl_between_slabs(
        substrate_slab=substrate_slab,
        film_slab=film_slab,
        gap=get_gap(row["substrate"], row["film"]),
        vacuum=INTERFACE_VACUUM,
        match_id=row["match index"],
        reduce_result_cell=False,
        **set_parameters_by_name[row["set"]]["matching_parameters"],
    )
    interface.name = f"{row['index in set']:03d} {interface.name}"
    interfaces_by_set[row["set"]].append(interface)
    print(f"{row['set']}: created {interface.name}")

## 4. Preview the interfaces

In [ ]:
for set_name, interfaces in interfaces_by_set.items():
    if not interfaces:
        continue
    print(set_name)
    interfaces_with_titles = [{"material": interface, "title": interface.name} for interface in interfaces]
    visualize(interfaces_with_titles, repetitions=[2, 2, 1], rotation="-90x")

## 5. Save the interface sets
Each set is written to `uploads/<set name>/`. To upload a set to the platform as a material set, open
[Create Material Set](utils_create_material_set.ipynb) and set `SUBFOLDER_NAME` to the set name.

In [ ]:
from mat3ra.notebooks_utils.material import set_materials
from mat3ra.notebooks_utils.settings import UPLOADS_FOLDER

for set_name, interfaces in interfaces_by_set.items():
    if interfaces:
        set_materials(interfaces, folder_path=f"{UPLOADS_FOLDER}/{set_name}")